# C / C++ 指標的指標 50 題：自動檢查版

本題本依照你提供的 C / C++ 自動檢查 Notebook 版型製作，主題集中在 **指標的指標**。

## 作答流程

1. 先執行下面的「C / C++ 自動檢查工具」cell。
2. 每題都有兩個作答 cell：
   - `%%c_answer Q...`：C（C11）
   - `%%cpp_answer Q...`：C++（C++17）
3. **不需要在作答 cell 手動輸入測試資料。**
4. 完成某一語言後，執行緊接在該作答 cell 下方的檢查。
5. 每題使用 **3 組測試資料**。
6. 最後執行「總檢查」查看完成度。

> 可以只練 C、只練 C++，或同一題兩種都寫。題目由基礎逐步增加整合程度。


In [ ]:
# === C / C++ 自動檢查工具：請先執行這個 cell，不要修改 ===
import base64
import json
import os
import shutil
import subprocess
import tempfile
from IPython.core.magic import register_cell_magic

_TEST_CASES = json.loads(
    base64.b64decode("eyJRMSI6IFt7ImlucHV0IjogIjEwXG4iLCAiZXhwZWN0ZWQiOiAiMTBcbiJ9LCB7ImlucHV0IjogIi00XG4iLCAiZXhwZWN0ZWQiOiAiLTRcbiJ9LCB7ImlucHV0IjogIjBcbiIsICJleHBlY3RlZCI6ICIwXG4ifV0sICJRMiI6IFt7ImlucHV0IjogIjNcbiIsICJleHBlY3RlZCI6ICI4XG4ifSwgeyJpbnB1dCI6ICItNVxuIiwgImV4cGVjdGVkIjogIjBcbiJ9LCB7ImlucHV0IjogIjBcbiIsICJleHBlY3RlZCI6ICI1XG4ifV0sICJRMyI6IFt7ImlucHV0IjogIjMgOVxuIiwgImV4cGVjdGVkIjogIjlcbiJ9LCB7ImlucHV0IjogIi0xIDVcbiIsICJleHBlY3RlZCI6ICI1XG4ifSwgeyJpbnB1dCI6ICI3IDdcbiIsICJleHBlY3RlZCI6ICI3XG4ifV0sICJRNCI6IFt7ImlucHV0IjogIjIgOFxuIiwgImV4cGVjdGVkIjogIjhcbiJ9LCB7ImlucHV0IjogIi0zIDRcbiIsICJleHBlY3RlZCI6ICI0XG4ifSwgeyJpbnB1dCI6ICI1IDVcbiIsICJleHBlY3RlZCI6ICI1XG4ifV0sICJRNSI6IFt7ImlucHV0IjogIjcgMTAwXG4iLCAiZXhwZWN0ZWQiOiAiMTAwXG4ifSwgeyJpbnB1dCI6ICItMiA5XG4iLCAiZXhwZWN0ZWQiOiAiOVxuIn0sIHsiaW5wdXQiOiAiMCAtNVxuIiwgImV4cGVjdGVkIjogIi01XG4ifV0sICJRNiI6IFt7ImlucHV0IjogIjEgOVxuIiwgImV4cGVjdGVkIjogIjkgMVxuIn0sIHsiaW5wdXQiOiAiLTIgNFxuIiwgImV4cGVjdGVkIjogIjQgLTJcbiJ9LCB7ImlucHV0IjogIjUgNVxuIiwgImV4cGVjdGVkIjogIjUgNVxuIn1dLCAiUTciOiBbeyJpbnB1dCI6ICIyIDhcbiIsICJleHBlY3RlZCI6ICI4XG4ifSwgeyJpbnB1dCI6ICI5IDNcbiIsICJleHBlY3RlZCI6ICI5XG4ifSwgeyJpbnB1dCI6ICI1IDVcbiIsICJleHBlY3RlZCI6ICI1XG4ifV0sICJROCI6IFt7ImlucHV0IjogIjIgOFxuIiwgImV4cGVjdGVkIjogIjJcbiJ9LCB7ImlucHV0IjogIjkgM1xuIiwgImV4cGVjdGVkIjogIjNcbiJ9LCB7ImlucHV0IjogIi00IC00XG4iLCAiZXhwZWN0ZWQiOiAiLTRcbiJ9XSwgIlE5IjogW3siaW5wdXQiOiAiNVxuIiwgImV4cGVjdGVkIjogIk5VTExcbiJ9LCB7ImlucHV0IjogIi0xXG4iLCAiZXhwZWN0ZWQiOiAiTlVMTFxuIn0sIHsiaW5wdXQiOiAiMFxuIiwgImV4cGVjdGVkIjogIk5VTExcbiJ9XSwgIlExMCI6IFt7ImlucHV0IjogIjMgOCAwXG4iLCAiZXhwZWN0ZWQiOiAiM1xuIn0sIHsiaW5wdXQiOiAiMyA4IDFcbiIsICJleHBlY3RlZCI6ICI4XG4ifSwgeyJpbnB1dCI6ICItMiA3IDFcbiIsICJleHBlY3RlZCI6ICI3XG4ifV0sICJRMTEiOiBbeyJpbnB1dCI6ICIyIDlcbiIsICJleHBlY3RlZCI6ICIyIDlcbiJ9LCB7ImlucHV0IjogIi0xIDRcbiIsICJleHBlY3RlZCI6ICItMSA0XG4ifSwgeyJpbnB1dCI6ICI3IDdcbiIsICJleHBlY3RlZCI6ICI3IDdcbiJ9XSwgIlExMiI6IFt7ImlucHV0IjogIjExXG4iLCAiZXhwZWN0ZWQiOiAiMTFcbiJ9LCB7ImlucHV0IjogIi02XG4iLCAiZXhwZWN0ZWQiOiAiLTZcbiJ9LCB7ImlucHV0IjogIjBcbiIsICJleHBlY3RlZCI6ICIwXG4ifV0sICJRMTMiOiBbeyJpbnB1dCI6ICI2XG4iLCAiZXhwZWN0ZWQiOiAiMTJcbiJ9LCB7ImlucHV0IjogIi0zXG4iLCAiZXhwZWN0ZWQiOiAiLTZcbiJ9LCB7ImlucHV0IjogIjBcbiIsICJleHBlY3RlZCI6ICIwXG4ifV0sICJRMTQiOiBbeyJpbnB1dCI6ICIxIDdcbiIsICJleHBlY3RlZCI6ICI3XG4ifSwgeyJpbnB1dCI6ICItNSAyXG4iLCAiZXhwZWN0ZWQiOiAiMlxuIn0sIHsiaW5wdXQiOiAiOSA5XG4iLCAiZXhwZWN0ZWQiOiAiOVxuIn1dLCAiUTE1IjogW3siaW5wdXQiOiAiNFxuIiwgImV4cGVjdGVkIjogIjE4XG4ifSwgeyJpbnB1dCI6ICIwXG4iLCAiZXhwZWN0ZWQiOiAiNlxuIn0sIHsiaW5wdXQiOiAiLTJcbiIsICJleHBlY3RlZCI6ICIwXG4ifV0sICJRMTYiOiBbeyJpbnB1dCI6ICIyIDggNVxuIiwgImV4cGVjdGVkIjogIjcgOFxuIn0sIHsiaW5wdXQiOiAiOSAzIDRcbiIsICJleHBlY3RlZCI6ICI5IDdcbiJ9LCB7ImlucHV0IjogIjUgNSAxXG4iLCAiZXhwZWN0ZWQiOiAiNiA1XG4ifV0sICJRMTciOiBbeyJpbnB1dCI6ICJhcHBsZSBiYW5hbmEgMFxuIiwgImV4cGVjdGVkIjogImFwcGxlXG4ifSwgeyJpbnB1dCI6ICJhcHBsZSBiYW5hbmEgMVxuIiwgImV4cGVjdGVkIjogImJhbmFuYVxuIn0sIHsiaW5wdXQiOiAieCB5eiA5XG4iLCAiZXhwZWN0ZWQiOiAieXpcbiJ9XSwgIlExOCI6IFt7ImlucHV0IjogImhlbGxvIHdvcmxkXG4iLCAiZXhwZWN0ZWQiOiAid29ybGQgaGVsbG9cbiJ9LCB7ImlucHV0IjogImEgYmJcbiIsICJleHBlY3RlZCI6ICJiYiBhXG4ifSwgeyJpbnB1dCI6ICJyZWQgYmx1ZVxuIiwgImV4cGVjdGVkIjogImJsdWUgcmVkXG4ifV0sICJRMTkiOiBbeyJpbnB1dCI6ICJjYXQgZWxlcGhhbnRcbiIsICJleHBlY3RlZCI6ICJlbGVwaGFudFxuIn0sIHsiaW5wdXQiOiAiYWFhIGJiXG4iLCAiZXhwZWN0ZWQiOiAiYWFhXG4ifSwgeyJpbnB1dCI6ICJhYiBjZFxuIiwgImV4cGVjdGVkIjogImFiXG4ifV0sICJRMjAiOiBbeyJpbnB1dCI6ICJhcHBsZSBiYW5hbmEgMFxuIiwgImV4cGVjdGVkIjogIkFwcGxlIGJhbmFuYVxuIn0sIHsiaW5wdXQiOiAiYXBwbGUgYmFuYW5hIDFcbiIsICJleHBlY3RlZCI6ICJhcHBsZSBCYW5hbmFcbiJ9LCB7ImlucHV0IjogInggeXogMVxuIiwgImV4cGVjdGVkIjogInggWXpcbiJ9XSwgIlEyMSI6IFt7ImlucHV0IjogIjEgMiAzXG4iLCAiZXhwZWN0ZWQiOiAiMSAyIDNcbiJ9LCB7ImlucHV0IjogIi0xIDUgOVxuIiwgImV4cGVjdGVkIjogIi0xIDUgOVxuIn0sIHsiaW5wdXQiOiAiNyA3IDhcbiIsICJleHBlY3RlZCI6ICI3IDcgOFxuIn1dLCAiUTIyIjogW3siaW5wdXQiOiAiMSAyIDMgNFxuIiwgImV4cGVjdGVkIjogIjEwXG4ifSwgeyJpbnB1dCI6ICItMSA1IDAgMlxuIiwgImV4cGVjdGVkIjogIjZcbiJ9LCB7ImlucHV0IjogIjEwIDEwIDEwIDEwXG4iLCAiZXhwZWN0ZWQiOiAiNDBcbiJ9XSwgIlEyMyI6IFt7ImlucHV0IjogIjEgMiAzXG4iLCAiZXhwZWN0ZWQiOiAiMyAyIDFcbiJ9LCB7ImlucHV0IjogIi0yIDAgOFxuIiwgImV4cGVjdGVkIjogIjggMCAtMlxuIn0sIHsiaW5wdXQiOiAiNSA2IDdcbiIsICJleHBlY3RlZCI6ICI3IDYgNVxuIn1dLCAiUTI0IjogW3siaW5wdXQiOiAiOSAyIDVcbiIsICJleHBlY3RlZCI6ICIyIDUgOVxuIn0sIHsiaW5wdXQiOiAiLTEgLTggM1xuIiwgImV4cGVjdGVkIjogIi04IC0xIDNcbiJ9LCB7ImlucHV0IjogIjQgNCAxXG4iLCAiZXhwZWN0ZWQiOiAiMSA0IDRcbiJ9XSwgIlEyNSI6IFt7ImlucHV0IjogIjEgMiAzXG4iLCAiZXhwZWN0ZWQiOiAiMTEgMTIgMTNcbiJ9LCB7ImlucHV0IjogIi0yIDAgOFxuIiwgImV4cGVjdGVkIjogIjggMTAgMThcbiJ9LCB7ImlucHV0IjogIjUgNSA1XG4iLCAiZXhwZWN0ZWQiOiAiMTUgMTUgMTVcbiJ9XSwgIlEyNiI6IFt7ImlucHV0IjogIjEgLTIgMyAwXG4iLCAiZXhwZWN0ZWQiOiAiMlxuIn0sIHsiaW5wdXQiOiAiLTEgLTIgLTMgLTRcbiIsICJleHBlY3RlZCI6ICIwXG4ifSwgeyJpbnB1dCI6ICI1IDYgNyA4XG4iLCAiZXhwZWN0ZWQiOiAiNFxuIn1dLCAiUTI3IjogW3siaW5wdXQiOiAiMSA5IDNcbiIsICJleHBlY3RlZCI6ICI5XG4ifSwgeyJpbnB1dCI6ICItOCAtMiAtNVxuIiwgImV4cGVjdGVkIjogIi0yXG4ifSwgeyJpbnB1dCI6ICI0IDQgNFxuIiwgImV4cGVjdGVkIjogIjRcbiJ9XSwgIlEyOCI6IFt7ImlucHV0IjogIjEgMiAzXG4iLCAiZXhwZWN0ZWQiOiAiM1xuIn0sIHsiaW5wdXQiOiAiLTEgNSA5XG4iLCAiZXhwZWN0ZWQiOiAiOVxuIn0sIHsiaW5wdXQiOiAiNyA3IDhcbiIsICJleHBlY3RlZCI6ICI4XG4ifV0sICJRMjkiOiBbeyJpbnB1dCI6ICIxIDIgMyA0IDUgNlxuIiwgImV4cGVjdGVkIjogIjIxXG4ifSwgeyJpbnB1dCI6ICItMSAwIDEgMiAzIDRcbiIsICJleHBlY3RlZCI6ICI5XG4ifSwgeyJpbnB1dCI6ICI1IDUgNSA1IDUgNVxuIiwgImV4cGVjdGVkIjogIjMwXG4ifV0sICJRMzAiOiBbeyJpbnB1dCI6ICIxIDIgMyA0IDUgNlxuIiwgImV4cGVjdGVkIjogIjYgMTVcbiJ9LCB7ImlucHV0IjogIi0xIDAgMSAyIDMgNFxuIiwgImV4cGVjdGVkIjogIjAgOVxuIn0sIHsiaW5wdXQiOiAiNSA1IDUgMSAxIDFcbiIsICJleHBlY3RlZCI6ICIxNSAzXG4ifV0sICJRMzEiOiBbeyJpbnB1dCI6ICJvbmUgdHdvIHRocmVlXG4iLCAiZXhwZWN0ZWQiOiAib25lIHR3byB0aHJlZVxuIn0sIHsiaW5wdXQiOiAiYSBiYiBjY2NcbiIsICJleHBlY3RlZCI6ICJhIGJiIGNjY1xuIn0sIHsiaW5wdXQiOiAicmVkIGdyZWVuIGJsdWVcbiIsICJleHBlY3RlZCI6ICJyZWQgZ3JlZW4gYmx1ZVxuIn1dLCAiUTMyIjogW3siaW5wdXQiOiAiYXBwbGUgYmFuYW5hIGNhdFxuIiwgImV4cGVjdGVkIjogImEgYiBjXG4ifSwgeyJpbnB1dCI6ICJSZWQgR3JlZW4gQmx1ZVxuIiwgImV4cGVjdGVkIjogIlIgRyBCXG4ifSwgeyJpbnB1dCI6ICJ4IHl6IHF3ZVxuIiwgImV4cGVjdGVkIjogInggeSBxXG4ifV0sICJRMzMiOiBbeyJpbnB1dCI6ICJjYXQgZWxlcGhhbnQgZG9nXG4iLCAiZXhwZWN0ZWQiOiAiZWxlcGhhbnRcbiJ9LCB7ImlucHV0IjogImFhYSBiYiBjXG4iLCAiZXhwZWN0ZWQiOiAiYWFhXG4ifSwgeyJpbnB1dCI6ICJ4IHl6IGFiY1xuIiwgImV4cGVjdGVkIjogImFiY1xuIn1dLCAiUTM0IjogW3siaW5wdXQiOiAib25lIHR3byB0aHJlZVxuIiwgImV4cGVjdGVkIjogInRocmVlIHR3byBvbmVcbiJ9LCB7ImlucHV0IjogImEgYmIgY2NjXG4iLCAiZXhwZWN0ZWQiOiAiY2NjIGJiIGFcbiJ9LCB7ImlucHV0IjogInJlZCBncmVlbiBibHVlXG4iLCAiZXhwZWN0ZWQiOiAiYmx1ZSBncmVlbiByZWRcbiJ9XSwgIlEzNSI6IFt7ImlucHV0IjogImFiYyBkZSBmXG4iLCAiZXhwZWN0ZWQiOiAiNlxuIn0sIHsiaW5wdXQiOiAiYSBiYiBjY2NcbiIsICJleHBlY3RlZCI6ICI2XG4ifSwgeyJpbnB1dCI6ICJwb2ludGVyIGMgY3BwXG4iLCAiZXhwZWN0ZWQiOiAiMTFcbiJ9XSwgIlEzNiI6IFt7ImlucHV0IjogIjEwXG4iLCAiZXhwZWN0ZWQiOiAiMTZcbiJ9LCB7ImlucHV0IjogIjBcbiIsICJleHBlY3RlZCI6ICI2XG4ifSwgeyJpbnB1dCI6ICItNVxuIiwgImV4cGVjdGVkIjogIjFcbiJ9XSwgIlEzNyI6IFt7ImlucHV0IjogIjRcbiIsICJleHBlY3RlZCI6ICIxMlxuIn0sIHsiaW5wdXQiOiAiLTJcbiIsICJleHBlY3RlZCI6ICItNlxuIn0sIHsiaW5wdXQiOiAiMFxuIiwgImV4cGVjdGVkIjogIjBcbiJ9XSwgIlEzOCI6IFt7ImlucHV0IjogIjIgOVxuIiwgImV4cGVjdGVkIjogIjlcbiJ9LCB7ImlucHV0IjogIi00IDFcbiIsICJleHBlY3RlZCI6ICIxXG4ifSwgeyJpbnB1dCI6ICI3IDdcbiIsICJleHBlY3RlZCI6ICI3XG4ifV0sICJRMzkiOiBbeyJpbnB1dCI6ICIxIDhcbiIsICJleHBlY3RlZCI6ICI4IDFcbiJ9LCB7ImlucHV0IjogIi0yIDRcbiIsICJleHBlY3RlZCI6ICI0IC0yXG4ifSwgeyJpbnB1dCI6ICI1IDVcbiIsICJleHBlY3RlZCI6ICI1IDVcbiJ9XSwgIlE0MCI6IFt7ImlucHV0IjogIjIgOCAwXG4iLCAiZXhwZWN0ZWQiOiAiMTIgOFxuIn0sIHsiaW5wdXQiOiAiMiA4IDFcbiIsICJleHBlY3RlZCI6ICIyIDE4XG4ifSwgeyJpbnB1dCI6ICItNSAxIDFcbiIsICJleHBlY3RlZCI6ICItNSAxMVxuIn1dLCAiUTQxIjogW3siaW5wdXQiOiAiMSA1IDMgNSA1XG4iLCAiZXhwZWN0ZWQiOiAiNVxuIn0sIHsiaW5wdXQiOiAiMSAyIDMgNCA5XG4iLCAiZXhwZWN0ZWQiOiAiLTFcbiJ9LCB7ImlucHV0IjogIi0yIDAgNyAxIC0yXG4iLCAiZXhwZWN0ZWQiOiAiLTJcbiJ9XSwgIlE0MiI6IFt7ImlucHV0IjogIjEgNSAzIDVcbiIsICJleHBlY3RlZCI6ICIxIDEwNSAzXG4ifSwgeyJpbnB1dCI6ICIxIDIgMyA5XG4iLCAiZXhwZWN0ZWQiOiAiMSAyIDNcbiJ9LCB7ImlucHV0IjogIjQgNCAyIDRcbiIsICJleHBlY3RlZCI6ICIxMDQgNCAyXG4ifV0sICJRNDMiOiBbeyJpbnB1dCI6ICIxMCAyMCAzMCAwIDJcbiIsICJleHBlY3RlZCI6ICIzMCAyMCAxMFxuIn0sIHsiaW5wdXQiOiAiMSAyIDMgMSAyXG4iLCAiZXhwZWN0ZWQiOiAiMSAzIDJcbiJ9LCB7ImlucHV0IjogIjUgNiA3IDAgMFxuIiwgImV4cGVjdGVkIjogIjUgNiA3XG4ifV0sICJRNDQiOiBbeyJpbnB1dCI6ICIzIDlcbiIsICJleHBlY3RlZCI6ICI5IDNcbiJ9LCB7ImlucHV0IjogIi0xIDRcbiIsICJleHBlY3RlZCI6ICI0IC0xXG4ifSwgeyJpbnB1dCI6ICI1IDVcbiIsICJleHBlY3RlZCI6ICI1IDVcbiJ9XSwgIlE0NSI6IFt7ImlucHV0IjogIjEgMiAzIDQgNSA2IDBcbiIsICJleHBlY3RlZCI6ICIxIDIgM1xuIn0sIHsiaW5wdXQiOiAiMSAyIDMgNCA1IDYgMVxuIiwgImV4cGVjdGVkIjogIjQgNSA2XG4ifSwgeyJpbnB1dCI6ICItMSAwIDEgMiAzIDQgMVxuIiwgImV4cGVjdGVkIjogIjIgMyA0XG4ifV0sICJRNDYiOiBbeyJpbnB1dCI6ICIxIDIgMyA0IDUgNlxuIiwgImV4cGVjdGVkIjogIjkgMTJcbiJ9LCB7ImlucHV0IjogIi0xIDIgMCAzIDQgLTVcbiIsICJleHBlY3RlZCI6ICIzIDBcbiJ9LCB7ImlucHV0IjogIjcgNyA3IDcgNyA3XG4iLCAiZXhwZWN0ZWQiOiAiMjEgMjFcbiJ9XSwgIlE0NyI6IFt7ImlucHV0IjogImFwcGxlIGJhbmFuYSBjYXQgMSAyXG4iLCAiZXhwZWN0ZWQiOiAiblxuIn0sIHsiaW5wdXQiOiAicmVkIGdyZWVuIGJsdWUgMiAwXG4iLCAiZXhwZWN0ZWQiOiAiYlxuIn0sIHsiaW5wdXQiOiAieCB5eiBxd2UgMCAwXG4iLCAiZXhwZWN0ZWQiOiAieFxuIn1dLCAiUTQ4IjogW3siaW5wdXQiOiAiYXBwbGUgc2t5IG9yYW5nZVxuIiwgImV4cGVjdGVkIjogIjZcbiJ9LCB7ImlucHV0IjogImJjZGYgZ2ggamtcbiIsICJleHBlY3RlZCI6ICIwXG4ifSwgeyJpbnB1dCI6ICJBRUlPVSBhIGVcbiIsICJleHBlY3RlZCI6ICI3XG4ifV0sICJRNDkiOiBbeyJpbnB1dCI6ICJwZWFyIGFwcGxlIG9yYW5nZVxuIiwgImV4cGVjdGVkIjogImFwcGxlIG9yYW5nZSBwZWFyXG4ifSwgeyJpbnB1dCI6ICJiIGEgY1xuIiwgImV4cGVjdGVkIjogImEgYiBjXG4ifSwgeyJpbnB1dCI6ICJhYSBhYSBhXG4iLCAiZXhwZWN0ZWQiOiAiYSBhYSBhYVxuIn1dLCAiUTUwIjogW3siaW5wdXQiOiAiMyA5IDFcbiIsICJleHBlY3RlZCI6ICIzIDkgOVxuIn0sIHsiaW5wdXQiOiAiNSAyIDhcbiIsICJleHBlY3RlZCI6ICI1IDggOFxuIn0sIHsiaW5wdXQiOiAiNCA0IDRcbiIsICJleHBlY3RlZCI6ICI0IDQgNFxuIn1dfQ==").decode("utf-8")
)

_ANSWERS = {"C": {}, "CPP": {}}
_RESULTS = {"C": {}, "CPP": {}}

def _normalise_output(text):
    lines = [line.rstrip() for line in text.replace("\r\n", "\n").split("\n")]
    while lines and lines[-1] == "":
        lines.pop()
    return "\n".join(lines)

def _compile_and_run(source, input_text, language):
    if language == "C":
        compiler = shutil.which("gcc")
        suffix = ".c"
        flags = ["-std=c11", "-O2"]
    else:
        compiler = shutil.which("g++")
        suffix = ".cpp"
        flags = ["-std=c++17", "-O2"]

    if compiler is None:
        return False, "", f"找不到 {'gcc' if language == 'C' else 'g++'}。請先安裝 C/C++ compiler。"

    with tempfile.TemporaryDirectory() as td:
        src = os.path.join(td, "answer" + suffix)
        exe = os.path.join(td, "answer")
        with open(src, "w", encoding="utf-8") as f:
            f.write(source)

        compile_result = subprocess.run(
            [compiler, src, "-o", exe] + flags,
            capture_output=True,
            text=True,
        )
        if compile_result.returncode != 0:
            return False, "", compile_result.stderr

        try:
            run_result = subprocess.run(
                [exe],
                input=input_text,
                capture_output=True,
                text=True,
                timeout=5,
            )
        except subprocess.TimeoutExpired:
            return False, "", "程式執行超過 5 秒，請檢查是否有無限迴圈。"

        if run_result.returncode != 0:
            err = run_result.stderr or f"程式結束碼：{run_result.returncode}"
            return False, run_result.stdout, err
        return True, run_result.stdout, ""

@register_cell_magic
def c_answer(line, cell):
    qid = line.strip().upper()
    _ANSWERS["C"][qid] = cell
    print(f"已儲存 {qid} 的 C 作答。請執行下方 C 檢查 cell。")

@register_cell_magic
def cpp_answer(line, cell):
    qid = line.strip().upper()
    _ANSWERS["CPP"][qid] = cell
    print(f"已儲存 {qid} 的 C++ 作答。請執行下方 C++ 檢查 cell。")

def _run_check(qid, language):
    qid = qid.upper()
    language = language.upper()
    label = "C" if language == "C" else "C++"

    if qid not in _ANSWERS[language]:
        print(f"❌ {qid}：尚未執行 {label} 作答 cell")
        _RESULTS[language][qid] = False
        return

    source = _ANSWERS[language][qid]
    cases = _TEST_CASES[qid]
    passed = 0

    for case_number, case in enumerate(cases, start=1):
        ok_run, stdout, error = _compile_and_run(source, case["input"], language)
        if not ok_run:
            print(f"❌ {qid}（{label}）：編譯或執行失敗")
            print(error)
            _RESULTS[language][qid] = False
            return

        actual = _normalise_output(stdout)
        expected = _normalise_output(case["expected"])

        if actual == expected:
            passed += 1
        else:
            print(f"❌ {qid}（{label}）測試 {case_number} 未通過")
            print("測試輸入：")
            print(case["input"], end="")
            print("你的輸出：")
            print(stdout if stdout else "<沒有輸出>")
            print("預期輸出：")
            print(case["expected"], end="")
            print()

    success = passed == len(cases)
    _RESULTS[language][qid] = success
    if success:
        print(f"✅ {qid}（{label}）全部通過（{passed} / {len(cases)}）")
    else:
        print(f"❌ {qid}（{label}）：通過 {passed} / {len(cases)} 個測試")

def _language_report(language):
    label = "C" if language == "C" else "C++"
    results = _RESULTS[language]
    passed = sum(results.get(f"Q{i}", False) for i in range(1, 51))
    untouched = [f"Q{i}" for i in range(1, 51) if f"Q{i}" not in results]
    print(f"{label}：{passed} / 50 題通過")
    if untouched:
        print("尚未執行檢查：" + ", ".join(untouched))

def _final_report():
    print("=== C / C++ 總檢查 ===")
    _language_report("C")
    print()
    _language_report("CPP")
    both_passed = sum(
        _RESULTS["C"].get(f"Q{i}", False)
        and _RESULTS["CPP"].get(f"Q{i}", False)
        for i in range(1, 51)
    )
    print()
    print(f"C 與 C++ 都通過：{both_passed} / 50 題")

print("C / C++ 自動檢查工具已載入。")
print("C 使用 C11；C++ 使用 C++17。")
print("每題有 3 組測試資料；作答後不需要手動輸入測試資料。")


## 作答方式提醒

每題流程：

```text
題目
↓
C 作答 cell ─────→ C 檢查
↓
C++ 作答 cell ───→ C++ 檢查
↓
同一組 3 組測試資料
↓
✅ 通過 / ❌ 顯示差異
```

只修改 `%%c_answer Q...` 與 `%%cpp_answer Q...` cell 裡的程式內容；不要修改檢查 cell。

**重要：** 自動檢查主要驗證輸出正確性；題目要求使用 pointer / array / function pointer 的部分，請自行遵守，才能真正練到該章觀念。


---

# Part A：int** 基本觀念（Q1–Q10）


## Q1. 讀取 **pp

讀入 x，建立 `p=&x`、`pp=&p`，輸出 `**pp`。

**練習重點：** double pointer basic

**範例輸入：**

```text
10
```

**範例輸出：**

```text
10
```


### C 作答

In [ ]:
%%c_answer Q1
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <ctype.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行 C 檢查：Q1

In [ ]:
# 檢查 Q1（C）：完成上方 C 程式後執行
_run_check("Q1", "C")


### C++ 作答

In [ ]:
%%cpp_answer Q1
#include <iostream>
#include <string>
#include <iomanip>
#include <algorithm>
#include <cctype>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行 C++ 檢查：Q1

In [ ]:
# 檢查 Q1（C++）：完成上方 C++ 程式後執行
_run_check("Q1", "CPP")


## Q2. 透過 **pp 加 5

讀入 x，透過 `**pp` 將 x 加 5 並輸出。

**練習重點：** double dereference write

**範例輸入：**

```text
3
```

**範例輸出：**

```text
8
```


### C 作答

In [ ]:
%%c_answer Q2
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <ctype.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行 C 檢查：Q2

In [ ]:
# 檢查 Q2（C）：完成上方 C 程式後執行
_run_check("Q2", "C")


### C++ 作答

In [ ]:
%%cpp_answer Q2
#include <iostream>
#include <string>
#include <iomanip>
#include <algorithm>
#include <cctype>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行 C++ 檢查：Q2

In [ ]:
# 檢查 Q2（C++）：完成上方 C++ 程式後執行
_run_check("Q2", "CPP")


## Q3. 透過 *pp 改變 p 的指向

讀入 a、b；p 先指向 a，pp 指向 p，再用 `*pp=&b`，輸出 `*p`。

**練習重點：** modify inner pointer

**範例輸入：**

```text
3 9
```

**範例輸出：**

```text
9
```


### C 作答

In [ ]:
%%c_answer Q3
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <ctype.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行 C 檢查：Q3

In [ ]:
# 檢查 Q3（C）：完成上方 C 程式後執行
_run_check("Q3", "C")


### C++ 作答

In [ ]:
%%cpp_answer Q3
#include <iostream>
#include <string>
#include <iomanip>
#include <algorithm>
#include <cctype>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行 C++ 檢查：Q3

In [ ]:
# 檢查 Q3（C++）：完成上方 C++ 程式後執行
_run_check("Q3", "CPP")


## Q4. setPointer 函式

撰寫 `setPointer(int **pp, int *target)`，讓呼叫者的 p 改指向 target。

**練習重點：** int** function parameter

**範例輸入：**

```text
2 8
```

**範例輸出：**

```text
8
```


### C 作答

In [ ]:
%%c_answer Q4
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <ctype.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行 C 檢查：Q4

In [ ]:
# 檢查 Q4（C）：完成上方 C 程式後執行
_run_check("Q4", "C")


### C++ 作答

In [ ]:
%%cpp_answer Q4
#include <iostream>
#include <string>
#include <iomanip>
#include <algorithm>
#include <cctype>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行 C++ 檢查：Q4

In [ ]:
# 檢查 Q4（C++）：完成上方 C++ 程式後執行
_run_check("Q4", "CPP")


## Q5. setValueThroughPP

撰寫函式接收 `int **pp, int value`，使用 `**pp` 修改底層整數。

**練習重點：** modify value through double pointer

**範例輸入：**

```text
7 100
```

**範例輸出：**

```text
100
```


### C 作答

In [ ]:
%%c_answer Q5
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <ctype.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行 C 檢查：Q5

In [ ]:
# 檢查 Q5（C）：完成上方 C 程式後執行
_run_check("Q5", "C")


### C++ 作答

In [ ]:
%%cpp_answer Q5
#include <iostream>
#include <string>
#include <iomanip>
#include <algorithm>
#include <cctype>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行 C++ 檢查：Q5

In [ ]:
# 檢查 Q5（C++）：完成上方 C++ 程式後執行
_run_check("Q5", "CPP")


## Q6. 交換兩個指標

讀入 a、b；pa=&a、pb=&b。撰寫 `swapPointers(int **x,int **y)` 只交換指標，最後輸出 `*pa *pb`。

**練習重點：** swap pointer variables

**範例輸入：**

```text
1 9
```

**範例輸出：**

```text
9 1
```


### C 作答

In [ ]:
%%c_answer Q6
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <ctype.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行 C 檢查：Q6

In [ ]:
# 檢查 Q6（C）：完成上方 C 程式後執行
_run_check("Q6", "C")


### C++ 作答

In [ ]:
%%cpp_answer Q6
#include <iostream>
#include <string>
#include <iomanip>
#include <algorithm>
#include <cctype>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行 C++ 檢查：Q6

In [ ]:
# 檢查 Q6（C++）：完成上方 C++ 程式後執行
_run_check("Q6", "CPP")


## Q7. out 指標選較大者

撰寫函式接收 a、b 的位址與 `int **out`，令 out 指向較大者，輸出 `**out`。

**練習重點：** double pointer output

**範例輸入：**

```text
2 8
```

**範例輸出：**

```text
8
```


### C 作答

In [ ]:
%%c_answer Q7
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <ctype.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行 C 檢查：Q7

In [ ]:
# 檢查 Q7（C）：完成上方 C 程式後執行
_run_check("Q7", "C")


### C++ 作答

In [ ]:
%%cpp_answer Q7
#include <iostream>
#include <string>
#include <iomanip>
#include <algorithm>
#include <cctype>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行 C++ 檢查：Q7

In [ ]:
# 檢查 Q7（C++）：完成上方 C++ 程式後執行
_run_check("Q7", "CPP")


## Q8. out 指標選較小者

同上，但 out 指向較小者。

**練習重點：** double pointer as output parameter

**範例輸入：**

```text
2 8
```

**範例輸出：**

```text
2
```


### C 作答

In [ ]:
%%c_answer Q8
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <ctype.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行 C 檢查：Q8

In [ ]:
# 檢查 Q8（C）：完成上方 C 程式後執行
_run_check("Q8", "C")


### C++ 作答

In [ ]:
%%cpp_answer Q8
#include <iostream>
#include <string>
#include <iomanip>
#include <algorithm>
#include <cctype>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行 C++ 檢查：Q8

In [ ]:
# 檢查 Q8（C++）：完成上方 C++ 程式後執行
_run_check("Q8", "CPP")


## Q9. 把 caller pointer 設為 NULL

p 原先指向 x。撰寫函式接收 `int **pp`，將 `*pp` 設為 NULL/nullptr。呼叫後輸出 `NULL`。

**練習重點：** double pointer changes pointer state

**範例輸入：**

```text
5
```

**範例輸出：**

```text
NULL
```


### C 作答

In [ ]:
%%c_answer Q9
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <ctype.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行 C 檢查：Q9

In [ ]:
# 檢查 Q9（C）：完成上方 C 程式後執行
_run_check("Q9", "C")


### C++ 作答

In [ ]:
%%cpp_answer Q9
#include <iostream>
#include <string>
#include <iomanip>
#include <algorithm>
#include <cctype>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行 C++ 檢查：Q9

In [ ]:
# 檢查 Q9（C++）：完成上方 C++ 程式後執行
_run_check("Q9", "CPP")


## Q10. 條件式重新指向

讀入 a、b、flag；p=&a，pp=&p。flag=1 時透過 pp 改指向 b。輸出 *p。

**練習重點：** conditional double-pointer reassignment

**範例輸入：**

```text
3 8 0
```

**範例輸出：**

```text
3
```


### C 作答

In [ ]:
%%c_answer Q10
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <ctype.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行 C 檢查：Q10

In [ ]:
# 檢查 Q10（C）：完成上方 C 程式後執行
_run_check("Q10", "C")


### C++ 作答

In [ ]:
%%cpp_answer Q10
#include <iostream>
#include <string>
#include <iomanip>
#include <algorithm>
#include <cctype>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行 C++ 檢查：Q10

In [ ]:
# 檢查 Q10（C++）：完成上方 C++ 程式後執行
_run_check("Q10", "CPP")


---

# Part B：多層解參考與 char**（Q11–Q20）


## Q11. pp 先指 p 再指 q

讀入 a、b；p=&a、q=&b、pp=&p。先輸出 **pp，再令 pp=&q 並輸出 **pp。

**練習重點：** double pointer variable itself can reassign

**範例輸入：**

```text
2 9
```

**範例輸出：**

```text
2 9
```


### C 作答

In [ ]:
%%c_answer Q11
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <ctype.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行 C 檢查：Q11

In [ ]:
# 檢查 Q11（C）：完成上方 C 程式後執行
_run_check("Q11", "C")


### C++ 作答

In [ ]:
%%cpp_answer Q11
#include <iostream>
#include <string>
#include <iomanip>
#include <algorithm>
#include <cctype>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行 C++ 檢查：Q11

In [ ]:
# 檢查 Q11（C++）：完成上方 C++ 程式後執行
_run_check("Q11", "CPP")


## Q12. 三重指標讀值

建立 x、p、pp、ppp，輸出 `***ppp`。

**練習重點：** triple pointer intro

**範例輸入：**

```text
11
```

**範例輸出：**

```text
11
```


### C 作答

In [ ]:
%%c_answer Q12
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <ctype.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行 C 檢查：Q12

In [ ]:
# 檢查 Q12（C）：完成上方 C 程式後執行
_run_check("Q12", "C")


### C++ 作答

In [ ]:
%%cpp_answer Q12
#include <iostream>
#include <string>
#include <iomanip>
#include <algorithm>
#include <cctype>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行 C++ 檢查：Q12

In [ ]:
# 檢查 Q12（C++）：完成上方 C++ 程式後執行
_run_check("Q12", "CPP")


## Q13. 三重指標修改值

透過 `***ppp` 將 x 乘 2。

**練習重點：** triple dereference

**範例輸入：**

```text
6
```

**範例輸出：**

```text
12
```


### C 作答

In [ ]:
%%c_answer Q13
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <ctype.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行 C 檢查：Q13

In [ ]:
# 檢查 Q13（C）：完成上方 C 程式後執行
_run_check("Q13", "C")


### C++ 作答

In [ ]:
%%cpp_answer Q13
#include <iostream>
#include <string>
#include <iomanip>
#include <algorithm>
#include <cctype>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行 C++ 檢查：Q13

In [ ]:
# 檢查 Q13（C++）：完成上方 C++ 程式後執行
_run_check("Q13", "CPP")


## Q14. 三重指標改變 p 指向

讀入 a、b。建立 p=&a、pp=&p、ppp=&pp；透過 `**ppp=&b` 讓 p 指向 b，輸出 *p。

**練習重點：** triple pointer chain

**範例輸入：**

```text
1 7
```

**範例輸出：**

```text
7
```


### C 作答

In [ ]:
%%c_answer Q14
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <ctype.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行 C 檢查：Q14

In [ ]:
# 檢查 Q14（C）：完成上方 C 程式後執行
_run_check("Q14", "C")


### C++ 作答

In [ ]:
%%cpp_answer Q14
#include <iostream>
#include <string>
#include <iomanip>
#include <algorithm>
#include <cctype>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行 C++ 檢查：Q14

In [ ]:
# 檢查 Q14（C++）：完成上方 C++ 程式後執行
_run_check("Q14", "CPP")


## Q15. 兩個 pp 修改同一 p

p 指向 x，pp1 與 pp2 都指向 p。用 `**pp1 += 2`、`**pp2 *= 3`，輸出 x。

**練習重點：** aliasing at double-pointer level

**範例輸入：**

```text
4
```

**範例輸出：**

```text
18
```


### C 作答

In [ ]:
%%c_answer Q15
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <ctype.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行 C 檢查：Q15

In [ ]:
# 檢查 Q15（C）：完成上方 C 程式後執行
_run_check("Q15", "C")


### C++ 作答

In [ ]:
%%cpp_answer Q15
#include <iostream>
#include <string>
#include <iomanip>
#include <algorithm>
#include <cctype>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行 C++ 檢查：Q15

In [ ]:
# 檢查 Q15（C++）：完成上方 C++ 程式後執行
_run_check("Q15", "CPP")


## Q16. choosePointer 函式後修改

讀入 a、b、k。函式透過 `int** out` 選較小者；main 再用 `**out += k` 修改它，輸出 a b。

**練習重點：** double-pointer selection + mutation

**範例輸入：**

```text
2 8 5
```

**範例輸出：**

```text
7 8
```


### C 作答

In [ ]:
%%c_answer Q16
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <ctype.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行 C 檢查：Q16

In [ ]:
# 檢查 Q16（C）：完成上方 C 程式後執行
_run_check("Q16", "C")


### C++ 作答

In [ ]:
%%cpp_answer Q16
#include <iostream>
#include <string>
#include <iomanip>
#include <algorithm>
#include <cctype>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行 C++ 檢查：Q16

In [ ]:
# 檢查 Q16（C++）：完成上方 C++ 程式後執行
_run_check("Q16", "CPP")


## Q17. char** 選擇兩字串

讀入兩個單字與 flag，建立兩個 char*，使用 `char** selected` 的概念選擇並輸出其中一個。

**練習重點：** char** basic

**範例輸入：**

```text
apple banana 0
```

**範例輸出：**

```text
apple
```


### C 作答

In [ ]:
%%c_answer Q17
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <ctype.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行 C 檢查：Q17

In [ ]:
# 檢查 Q17（C）：完成上方 C 程式後執行
_run_check("Q17", "C")


### C++ 作答

In [ ]:
%%cpp_answer Q17
#include <iostream>
#include <string>
#include <iomanip>
#include <algorithm>
#include <cctype>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行 C++ 檢查：Q17

In [ ]:
# 檢查 Q17（C++）：完成上方 C++ 程式後執行
_run_check("Q17", "CPP")


## Q18. 交換兩個 char*

讀入兩個單字到 char arrays，建立 p1、p2，使用 `char**` swap function 只交換指標，再輸出。

**練習重點：** char** pointer swap

**範例輸入：**

```text
hello world
```

**範例輸出：**

```text
world hello
```


### C 作答

In [ ]:
%%c_answer Q18
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <ctype.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行 C 檢查：Q18

In [ ]:
# 檢查 Q18（C）：完成上方 C 程式後執行
_run_check("Q18", "C")


### C++ 作答

In [ ]:
%%cpp_answer Q18
#include <iostream>
#include <string>
#include <iomanip>
#include <algorithm>
#include <cctype>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行 C++ 檢查：Q18

In [ ]:
# 檢查 Q18（C++）：完成上方 C++ 程式後執行
_run_check("Q18", "CPP")


## Q19. char** 選較長字串

讀入兩單字，用函式透過 `char** out` 指向較長者；同長取第一個，輸出字串。

**練習重點：** char** output parameter

**範例輸入：**

```text
cat elephant
```

**範例輸出：**

```text
elephant
```


### C 作答

In [ ]:
%%c_answer Q19
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <ctype.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行 C 檢查：Q19

In [ ]:
# 檢查 Q19（C）：完成上方 C 程式後執行
_run_check("Q19", "C")


### C++ 作答

In [ ]:
%%cpp_answer Q19
#include <iostream>
#include <string>
#include <iomanip>
#include <algorithm>
#include <cctype>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行 C++ 檢查：Q19

In [ ]:
# 檢查 Q19（C++）：完成上方 C++ 程式後執行
_run_check("Q19", "CPP")


## Q20. char** 修改被選字串首字母

讀入兩個小寫單字與 flag，透過 char** 選定其中一個 char*，把其首字母改成大寫，輸出兩字串。

**練習重點：** double pointer + char mutation

**範例輸入：**

```text
apple banana 0
```

**範例輸出：**

```text
Apple banana
```


### C 作答

In [ ]:
%%c_answer Q20
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <ctype.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行 C 檢查：Q20

In [ ]:
# 檢查 Q20（C）：完成上方 C 程式後執行
_run_check("Q20", "C")


### C++ 作答

In [ ]:
%%cpp_answer Q20
#include <iostream>
#include <string>
#include <iomanip>
#include <algorithm>
#include <cctype>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行 C++ 檢查：Q20

In [ ]:
# 檢查 Q20（C++）：完成上方 C++ 程式後執行
_run_check("Q20", "CPP")


---

# Part C：Pointer Array + int**（Q21–Q30）


## Q21. int* 陣列透過 int** 讀取

讀入三個整數，建立 `int *ptrs[3]`，令 `int **pp=ptrs`，輸出 `**pp **(pp+1) **(pp+2)`。

**練習重點：** array of pointers decays to int**

**範例輸入：**

```text
1 2 3
```

**範例輸出：**

```text
1 2 3
```


### C 作答

In [ ]:
%%c_answer Q21
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <ctype.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行 C 檢查：Q21

In [ ]:
# 檢查 Q21（C）：完成上方 C 程式後執行
_run_check("Q21", "C")


### C++ 作答

In [ ]:
%%cpp_answer Q21
#include <iostream>
#include <string>
#include <iomanip>
#include <algorithm>
#include <cctype>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行 C++ 檢查：Q21

In [ ]:
# 檢查 Q21（C++）：完成上方 C++ 程式後執行
_run_check("Q21", "CPP")


## Q22. int** 遍歷求和

讀入四數，建立 pointer array，再使用 `int**` 走訪求和。

**練習重點：** int** traversal

**範例輸入：**

```text
1 2 3 4
```

**範例輸出：**

```text
10
```


### C 作答

In [ ]:
%%c_answer Q22
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <ctype.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行 C 檢查：Q22

In [ ]:
# 檢查 Q22（C）：完成上方 C 程式後執行
_run_check("Q22", "C")


### C++ 作答

In [ ]:
%%cpp_answer Q22
#include <iostream>
#include <string>
#include <iomanip>
#include <algorithm>
#include <cctype>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行 C++ 檢查：Q22

In [ ]:
# 檢查 Q22（C++）：完成上方 C++ 程式後執行
_run_check("Q22", "CPP")


## Q23. int** 反向讀取

建立三個 int* 的陣列，使用 `int**` 從尾向前解參考輸出。

**練習重點：** pointer-to-pointer arithmetic

**範例輸入：**

```text
1 2 3
```

**範例輸出：**

```text
3 2 1
```


### C 作答

In [ ]:
%%c_answer Q23
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <ctype.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行 C 檢查：Q23

In [ ]:
# 檢查 Q23（C）：完成上方 C 程式後執行
_run_check("Q23", "C")


### C++ 作答

In [ ]:
%%cpp_answer Q23
#include <iostream>
#include <string>
#include <iomanip>
#include <algorithm>
#include <cctype>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行 C++ 檢查：Q23

In [ ]:
# 檢查 Q23（C++）：完成上方 C++ 程式後執行
_run_check("Q23", "CPP")


## Q24. 排序 pointer array

讀入三數，只交換 `int* ptrs[]` 中的指標，使 `**` 讀出的值升冪，不改原變數。

**練習重點：** sorting pointers through int**

**範例輸入：**

```text
9 2 5
```

**範例輸出：**

```text
2 5 9
```


### C 作答

In [ ]:
%%c_answer Q24
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <ctype.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行 C 檢查：Q24

In [ ]:
# 檢查 Q24（C）：完成上方 C 程式後執行
_run_check("Q24", "C")


### C++ 作答

In [ ]:
%%cpp_answer Q24
#include <iostream>
#include <string>
#include <iomanip>
#include <algorithm>
#include <cctype>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行 C++ 檢查：Q24

In [ ]:
# 檢查 Q24（C++）：完成上方 C++ 程式後執行
_run_check("Q24", "CPP")


## Q25. 透過 int** 全部加 10

讀入三數，以 pointer array + int** 將三個原變數都加 10。

**練習重點：** double dereference in pointer array

**範例輸入：**

```text
1 2 3
```

**範例輸出：**

```text
11 12 13
```


### C 作答

In [ ]:
%%c_answer Q25
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <ctype.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行 C 檢查：Q25

In [ ]:
# 檢查 Q25（C）：完成上方 C 程式後執行
_run_check("Q25", "C")


### C++ 作答

In [ ]:
%%cpp_answer Q25
#include <iostream>
#include <string>
#include <iomanip>
#include <algorithm>
#include <cctype>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行 C++ 檢查：Q25

In [ ]:
# 檢查 Q25（C++）：完成上方 C++ 程式後執行
_run_check("Q25", "CPP")


## Q26. int** 計算正數數量

讀入四數，以 pointer array 與 int** 遍歷，計算解參考後 >0 的數量。

**練習重點：** double pointer traversal

**範例輸入：**

```text
1 -2 3 0
```

**範例輸出：**

```text
2
```


### C 作答

In [ ]:
%%c_answer Q26
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <ctype.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行 C 檢查：Q26

In [ ]:
# 檢查 Q26（C）：完成上方 C 程式後執行
_run_check("Q26", "C")


### C++ 作答

In [ ]:
%%cpp_answer Q26
#include <iostream>
#include <string>
#include <iomanip>
#include <algorithm>
#include <cctype>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行 C++ 檢查：Q26

In [ ]:
# 檢查 Q26（C++）：完成上方 C++ 程式後執行
_run_check("Q26", "CPP")


## Q27. 選出最大值指標的位址

pointer array 中找最大值那一個元素位置，令 `int **best` 指向該 pointer slot，輸出 `**best`。

**練習重點：** pointer to pointer slot

**範例輸入：**

```text
1 9 3
```

**範例輸出：**

```text
9
```


### C 作答

In [ ]:
%%c_answer Q27
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <ctype.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行 C 檢查：Q27

In [ ]:
# 檢查 Q27（C）：完成上方 C 程式後執行
_run_check("Q27", "C")


### C++ 作答

In [ ]:
%%cpp_answer Q27
#include <iostream>
#include <string>
#include <iomanip>
#include <algorithm>
#include <cctype>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行 C++ 檢查：Q27

In [ ]:
# 檢查 Q27（C++）：完成上方 C++ 程式後執行
_run_check("Q27", "CPP")


## Q28. 修改 pointer slot

讀入 a、b、c。ptrs={&a,&b,&c}；令 pp=&ptrs[0]，再把 `*pp` 改成 &c，輸出 `**pp`。

**練習重點：** changing an element of pointer array via int**

**範例輸入：**

```text
1 2 3
```

**範例輸出：**

```text
3
```


### C 作答

In [ ]:
%%c_answer Q28
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <ctype.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行 C 檢查：Q28

In [ ]:
# 檢查 Q28（C）：完成上方 C 程式後執行
_run_check("Q28", "C")


### C++ 作答

In [ ]:
%%cpp_answer Q28
#include <iostream>
#include <string>
#include <iomanip>
#include <algorithm>
#include <cctype>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行 C++ 檢查：Q28

In [ ]:
# 檢查 Q28（C++）：完成上方 C++ 程式後執行
_run_check("Q28", "CPP")


## Q29. 2×3 row pointers 求總和

讀入 6 整數形成兩個長度 3 的 row arrays，建立 `int *rows[2]` 與 `int **pp=rows`，透過 `pp` 求全部總和。

**練習重點：** row-pointer table

**範例輸入：**

```text
1 2 3 4 5 6
```

**範例輸出：**

```text
21
```


### C 作答

In [ ]:
%%c_answer Q29
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <ctype.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行 C 檢查：Q29

In [ ]:
# 檢查 Q29（C）：完成上方 C 程式後執行
_run_check("Q29", "C")


### C++ 作答

In [ ]:
%%cpp_answer Q29
#include <iostream>
#include <string>
#include <iomanip>
#include <algorithm>
#include <cctype>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行 C++ 檢查：Q29

In [ ]:
# 檢查 Q29（C++）：完成上方 C++ 程式後執行
_run_check("Q29", "CPP")


## Q30. 2×3 row sums

同樣建立 row pointers，輸出兩列各自總和。

**練習重點：** int** row access

**範例輸入：**

```text
1 2 3 4 5 6
```

**範例輸出：**

```text
6 15
```


### C 作答

In [ ]:
%%c_answer Q30
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <ctype.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行 C 檢查：Q30

In [ ]:
# 檢查 Q30（C）：完成上方 C 程式後執行
_run_check("Q30", "C")


### C++ 作答

In [ ]:
%%cpp_answer Q30
#include <iostream>
#include <string>
#include <iomanip>
#include <algorithm>
#include <cctype>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行 C++ 檢查：Q30

In [ ]:
# 檢查 Q30（C++）：完成上方 C++ 程式後執行
_run_check("Q30", "CPP")


---

# Part D：char** 與 int***（Q31–Q40）


## Q31. char* 陣列透過 char** 輸出

讀入三個單字，建立 `char* words[3]`、`char** pp=words`，依序輸出三字串。

**練習重點：** char** traversal

**範例輸入：**

```text
one two three
```

**範例輸出：**

```text
one two three
```


### C 作答

In [ ]:
%%c_answer Q31
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <ctype.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行 C 檢查：Q31

In [ ]:
# 檢查 Q31（C）：完成上方 C 程式後執行
_run_check("Q31", "C")


### C++ 作答

In [ ]:
%%cpp_answer Q31
#include <iostream>
#include <string>
#include <iomanip>
#include <algorithm>
#include <cctype>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行 C++ 檢查：Q31

In [ ]:
# 檢查 Q31（C++）：完成上方 C++ 程式後執行
_run_check("Q31", "CPP")


## Q32. char** 首字母

讀入三單字，用 `char**` 遍歷，輸出各自首字母。

**練習重點：** double dereference with strings

**範例輸入：**

```text
apple banana cat
```

**範例輸出：**

```text
a b c
```


### C 作答

In [ ]:
%%c_answer Q32
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <ctype.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行 C 檢查：Q32

In [ ]:
# 檢查 Q32（C）：完成上方 C 程式後執行
_run_check("Q32", "C")


### C++ 作答

In [ ]:
%%cpp_answer Q32
#include <iostream>
#include <string>
#include <iomanip>
#include <algorithm>
#include <cctype>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行 C++ 檢查：Q32

In [ ]:
# 檢查 Q32（C++）：完成上方 C++ 程式後執行
_run_check("Q32", "CPP")


## Q33. char** 找最長字串

讀入三單字，以 char** 找最長者，同長取最前。

**練習重點：** char** selection

**範例輸入：**

```text
cat elephant dog
```

**範例輸出：**

```text
elephant
```


### C 作答

In [ ]:
%%c_answer Q33
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <ctype.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行 C 檢查：Q33

In [ ]:
# 檢查 Q33（C）：完成上方 C 程式後執行
_run_check("Q33", "C")


### C++ 作答

In [ ]:
%%cpp_answer Q33
#include <iostream>
#include <string>
#include <iomanip>
#include <algorithm>
#include <cctype>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行 C++ 檢查：Q33

In [ ]:
# 檢查 Q33（C++）：完成上方 C++ 程式後執行
_run_check("Q33", "CPP")


## Q34. char** 交換第一與最後 pointer

不改字串內容，只交換 words[0] 與 words[2] 的指標，輸出三字串。

**練習重點：** pointer slots via char**

**範例輸入：**

```text
one two three
```

**範例輸出：**

```text
three two one
```


### C 作答

In [ ]:
%%c_answer Q34
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <ctype.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行 C 檢查：Q34

In [ ]:
# 檢查 Q34（C）：完成上方 C 程式後執行
_run_check("Q34", "C")


### C++ 作答

In [ ]:
%%cpp_answer Q34
#include <iostream>
#include <string>
#include <iomanip>
#include <algorithm>
#include <cctype>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行 C++ 檢查：Q34

In [ ]:
# 檢查 Q34（C++）：完成上方 C++ 程式後執行
_run_check("Q34", "CPP")


## Q35. char** 總字元數

讀入三單字，使用 char** 逐字串計算總長度。

**練習重點：** char** + inner char*

**範例輸入：**

```text
abc de f
```

**範例輸出：**

```text
6
```


### C 作答

In [ ]:
%%c_answer Q35
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <ctype.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行 C 檢查：Q35

In [ ]:
# 檢查 Q35（C）：完成上方 C 程式後執行
_run_check("Q35", "C")


### C++ 作答

In [ ]:
%%cpp_answer Q35
#include <iostream>
#include <string>
#include <iomanip>
#include <algorithm>
#include <cctype>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行 C++ 檢查：Q35

In [ ]:
# 檢查 Q35（C++）：完成上方 C++ 程式後執行
_run_check("Q35", "CPP")


## Q36. 三重指標鏈

讀入 x，建立 `int *p`、`int **pp`、`int ***ppp`，依序讓 `***ppp += 1`、`**pp += 2`、`*p += 3`，輸出 x。

**練習重點：** multi-level aliasing

**範例輸入：**

```text
10
```

**範例輸出：**

```text
16
```


### C 作答

In [ ]:
%%c_answer Q36
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <ctype.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行 C 檢查：Q36

In [ ]:
# 檢查 Q36（C）：完成上方 C 程式後執行
_run_check("Q36", "C")


### C++ 作答

In [ ]:
%%cpp_answer Q36
#include <iostream>
#include <string>
#include <iomanip>
#include <algorithm>
#include <cctype>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行 C++ 檢查：Q36

In [ ]:
# 檢查 Q36（C++）：完成上方 C++ 程式後執行
_run_check("Q36", "CPP")


## Q37. 函式接收 int*** 修改底層

撰寫函式接收 `int ***ppp`，把 `***ppp` 乘 3。

**練習重點：** triple pointer parameter

**範例輸入：**

```text
4
```

**範例輸出：**

```text
12
```


### C 作答

In [ ]:
%%c_answer Q37
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <ctype.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行 C 檢查：Q37

In [ ]:
# 檢查 Q37（C）：完成上方 C 程式後執行
_run_check("Q37", "C")


### C++ 作答

In [ ]:
%%cpp_answer Q37
#include <iostream>
#include <string>
#include <iomanip>
#include <algorithm>
#include <cctype>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行 C++ 檢查：Q37

In [ ]:
# 檢查 Q37（C++）：完成上方 C++ 程式後執行
_run_check("Q37", "CPP")


## Q38. 函式用 int*** 改變 p

讀入 a、b；建立 p=&a、pp=&p。函式接收 `int***` 與 &b，讓最終 p 指向 b，輸出 *p。

**練習重點：** modify nested pointer via int***

**範例輸入：**

```text
2 9
```

**範例輸出：**

```text
9
```


### C 作答

In [ ]:
%%c_answer Q38
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <ctype.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行 C 檢查：Q38

In [ ]:
# 檢查 Q38（C）：完成上方 C 程式後執行
_run_check("Q38", "C")


### C++ 作答

In [ ]:
%%cpp_answer Q38
#include <iostream>
#include <string>
#include <iomanip>
#include <algorithm>
#include <cctype>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行 C++ 檢查：Q38

In [ ]:
# 檢查 Q38（C++）：完成上方 C++ 程式後執行
_run_check("Q38", "CPP")


## Q39. 交換兩個 int** 指向

讀入 a、b；p=&a、q=&b；pp=&p、qq=&q。交換 pp 與 qq 本身，輸出 `**pp **qq`。

**練習重點：** pointer-to-pointer variables

**範例輸入：**

```text
1 8
```

**範例輸出：**

```text
8 1
```


### C 作答

In [ ]:
%%c_answer Q39
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <ctype.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行 C 檢查：Q39

In [ ]:
# 檢查 Q39（C）：完成上方 C 程式後執行
_run_check("Q39", "C")


### C++ 作答

In [ ]:
%%cpp_answer Q39
#include <iostream>
#include <string>
#include <iomanip>
#include <algorithm>
#include <cctype>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行 C++ 檢查：Q39

In [ ]:
# 檢查 Q39（C++）：完成上方 C++ 程式後執行
_run_check("Q39", "CPP")


## Q40. 雙層選擇鏈

讀入 a、b、flag。p=&a、q=&b；chosen 是 `int**`，flag=0 令 chosen=&p，否則 chosen=&q；透過 `**chosen += 10`，輸出 a b。

**練習重點：** select a pointer variable via int**

**範例輸入：**

```text
2 8 0
```

**範例輸出：**

```text
12 8
```


### C 作答

In [ ]:
%%c_answer Q40
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <ctype.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行 C 檢查：Q40

In [ ]:
# 檢查 Q40（C）：完成上方 C 程式後執行
_run_check("Q40", "C")


### C++ 作答

In [ ]:
%%cpp_answer Q40
#include <iostream>
#include <string>
#include <iomanip>
#include <algorithm>
#include <cctype>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行 C++ 檢查：Q40

In [ ]:
# 檢查 Q40（C++）：完成上方 C++ 程式後執行
_run_check("Q40", "CPP")


---

# Part E：Pointer Slot / Row Pointer 綜合（Q41–Q50）


## Q41. findPointerSlot

讀入四個整數與 target。建立 pointer array；函式回傳 `int**` 指向第一個值等於 target 的 pointer slot；找不到輸出 -1，找到輸出 `**slot`。

**練習重點：** returning pointer-to-pointer

**範例輸入：**

```text
1 5 3 5 5
```

**範例輸出：**

```text
5
```


### C 作答

In [ ]:
%%c_answer Q41
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <ctype.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行 C 檢查：Q41

In [ ]:
# 檢查 Q41（C）：完成上方 C 程式後執行
_run_check("Q41", "C")


### C++ 作答

In [ ]:
%%cpp_answer Q41
#include <iostream>
#include <string>
#include <iomanip>
#include <algorithm>
#include <cctype>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行 C++ 檢查：Q41

In [ ]:
# 檢查 Q41（C++）：完成上方 C++ 程式後執行
_run_check("Q41", "CPP")


## Q42. 透過返回的 slot 修改原值

讀入三數與 target。找到 pointer slot 後，用 `**slot += 100`；找不到不改。輸出三數。

**練習重點：** int** returned slot + mutation

**範例輸入：**

```text
1 5 3 5
```

**範例輸出：**

```text
1 105 3
```


### C 作答

In [ ]:
%%c_answer Q42
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <ctype.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行 C 檢查：Q42

In [ ]:
# 檢查 Q42（C）：完成上方 C 程式後執行
_run_check("Q42", "C")


### C++ 作答

In [ ]:
%%cpp_answer Q42
#include <iostream>
#include <string>
#include <iomanip>
#include <algorithm>
#include <cctype>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行 C++ 檢查：Q42

In [ ]:
# 檢查 Q42（C++）：完成上方 C++ 程式後執行
_run_check("Q42", "CPP")


## Q43. 兩個 pointer slots 交換

讀入三數與 i、j（0..2），利用 `int**` 指向 ptrs[i]、ptrs[j]，交換兩個 slot 裡的指標，再依 ptrs 順序輸出值。

**練習重點：** swap pointer-array slots through int**

**範例輸入：**

```text
10 20 30 0 2
```

**範例輸出：**

```text
30 20 10
```


### C 作答

In [ ]:
%%c_answer Q43
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <ctype.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行 C 檢查：Q43

In [ ]:
# 檢查 Q43（C）：完成上方 C 程式後執行
_run_check("Q43", "C")


### C++ 作答

In [ ]:
%%cpp_answer Q43
#include <iostream>
#include <string>
#include <iomanip>
#include <algorithm>
#include <cctype>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行 C++ 檢查：Q43

In [ ]:
# 檢查 Q43（C++）：完成上方 C++ 程式後執行
_run_check("Q43", "CPP")


## Q44. 間接交換底層值

a、b 各有 pointer，pp、qq 再指向這些 pointer；使用 `**pp` 與 `**qq` 交換底層 a、b。

**練習重點：** double dereference to values

**範例輸入：**

```text
3 9
```

**範例輸出：**

```text
9 3
```


### C 作答

In [ ]:
%%c_answer Q44
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <ctype.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行 C 檢查：Q44

In [ ]:
# 檢查 Q44（C）：完成上方 C 程式後執行
_run_check("Q44", "C")


### C++ 作答

In [ ]:
%%cpp_answer Q44
#include <iostream>
#include <string>
#include <iomanip>
#include <algorithm>
#include <cctype>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行 C++ 檢查：Q44

In [ ]:
# 檢查 Q44（C++）：完成上方 C++ 程式後執行
_run_check("Q44", "CPP")


## Q45. row pointer 選列

讀入 2×3 六數與 row index，建立 rows[]；令 `int **selected=&rows[index]`，輸出該列三數。

**練習重點：** int** selects row pointer

**範例輸入：**

```text
1 2 3 4 5 6 0
```

**範例輸出：**

```text
1 2 3
```


### C 作答

In [ ]:
%%c_answer Q45
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <ctype.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行 C 檢查：Q45

In [ ]:
# 檢查 Q45（C）：完成上方 C 程式後執行
_run_check("Q45", "C")


### C++ 作答

In [ ]:
%%cpp_answer Q45
#include <iostream>
#include <string>
#include <iomanip>
#include <algorithm>
#include <cctype>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行 C++ 檢查：Q45

In [ ]:
# 檢查 Q45（C++）：完成上方 C++ 程式後執行
_run_check("Q45", "CPP")


## Q46. row pointers 欄總和

建立 3×2 三個 row arrays 與 `int** rows`，輸出兩欄總和。

**練習重點：** int** row table

**範例輸入：**

```text
1 2 3 4 5 6
```

**範例輸出：**

```text
9 12
```


### C 作答

In [ ]:
%%c_answer Q46
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <ctype.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行 C 檢查：Q46

In [ ]:
# 檢查 Q46（C）：完成上方 C 程式後執行
_run_check("Q46", "C")


### C++ 作答

In [ ]:
%%cpp_answer Q46
#include <iostream>
#include <string>
#include <iomanip>
#include <algorithm>
#include <cctype>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行 C++ 檢查：Q46

In [ ]:
# 檢查 Q46（C++）：完成上方 C++ 程式後執行
_run_check("Q46", "CPP")


## Q47. char** 選單字再選字元

讀入三單字、word index、char index，使用 char** 選到字串後輸出指定字元。

**練習重點：** char** + char* arithmetic

**範例輸入：**

```text
apple banana cat 1 2
```

**範例輸出：**

```text
n
```


### C 作答

In [ ]:
%%c_answer Q47
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <ctype.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行 C 檢查：Q47

In [ ]:
# 檢查 Q47（C）：完成上方 C 程式後執行
_run_check("Q47", "C")


### C++ 作答

In [ ]:
%%cpp_answer Q47
#include <iostream>
#include <string>
#include <iomanip>
#include <algorithm>
#include <cctype>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行 C++ 檢查：Q47

In [ ]:
# 檢查 Q47（C++）：完成上方 C++ 程式後執行
_run_check("Q47", "CPP")


## Q48. char** 全部母音數

讀入三單字，以 char** + char* 雙層走訪計算所有母音。

**練習重點：** nested pointer traversal

**範例輸入：**

```text
apple sky orange
```

**範例輸出：**

```text
6
```


### C 作答

In [ ]:
%%c_answer Q48
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <ctype.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行 C 檢查：Q48

In [ ]:
# 檢查 Q48（C）：完成上方 C 程式後執行
_run_check("Q48", "C")


### C++ 作答

In [ ]:
%%cpp_answer Q48
#include <iostream>
#include <string>
#include <iomanip>
#include <algorithm>
#include <cctype>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行 C++ 檢查：Q48

In [ ]:
# 檢查 Q48（C++）：完成上方 C++ 程式後執行
_run_check("Q48", "CPP")


## Q49. 只交換字串指標排序

讀入三單字，依字典序升冪排列，但只交換 `char*` 指標，不搬移字串內容。

**練習重點：** sorting char* slots

**範例輸入：**

```text
pear apple orange
```

**範例輸出：**

```text
apple orange pear
```


### C 作答

In [ ]:
%%c_answer Q49
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <ctype.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行 C 檢查：Q49

In [ ]:
# 檢查 Q49（C）：完成上方 C 程式後執行
_run_check("Q49", "C")


### C++ 作答

In [ ]:
%%cpp_answer Q49
#include <iostream>
#include <string>
#include <iomanip>
#include <algorithm>
#include <cctype>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行 C++ 檢查：Q49

In [ ]:
# 檢查 Q49（C++）：完成上方 C++ 程式後執行
_run_check("Q49", "CPP")


## Q50. Pointer to Pointer 總整合

讀入三個整數。建立 pointer array 與 `int**`；找最小值的 pointer slot，把該 slot 改成指向最大值，再輸出 ptrs 解參考後三個值。若最小/最大有重複，取第一次。

**練習重點：** int** slot manipulation 綜合

**範例輸入：**

```text
3 9 1
```

**範例輸出：**

```text
3 9 9
```


### C 作答

In [ ]:
%%c_answer Q50
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <ctype.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行 C 檢查：Q50

In [ ]:
# 檢查 Q50（C）：完成上方 C 程式後執行
_run_check("Q50", "C")


### C++ 作答

In [ ]:
%%cpp_answer Q50
#include <iostream>
#include <string>
#include <iomanip>
#include <algorithm>
#include <cctype>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行 C++ 檢查：Q50

In [ ]:
# 檢查 Q50（C++）：完成上方 C++ 程式後執行
_run_check("Q50", "CPP")


---

# 總檢查


In [ ]:
_final_report()
